*0.1 Python for GenAI · async/await, asyncio.gather, concurrency for API calls*

# asyncio.gather

**The situation.** Every night your company indexes the day's support tickets so they can be searched. Each ticket is sent to the embedding model. Tonight there are 1,000 tickets. One call takes about 0.3 seconds.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

import asyncio
from concurrent.futures import ThreadPoolExecutor


# A notebook already has an event loop running, so asyncio.run() is not allowed here.
# This helper runs the async code on a separate thread instead. In a normal script you
# would simply write asyncio.run(main()).
def run_async(coroutine):
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()

**The job written the obvious way.** A loop: send one ticket, wait for the result, send the next. This is how almost everyone writes it first.

In [2]:
import time

from openai import AsyncOpenAI

# The tickets. In production they come from a database; here they are generated.
tickets = []
for number in range(1, 41):
    tickets.append(f"Ticket {number}: customer cannot log in after password reset")


async def embed_one(ai: AsyncOpenAI, ticket: str) -> int:
    reply = await ai.embeddings.create(model="text-embedding-3-small", input=ticket)
    return len(reply.data[0].embedding)


async def one_by_one() -> int:
    async with AsyncOpenAI(timeout=30) as ai:
        done = 0
        for ticket in tickets:
            await embed_one(ai, ticket)  # wait for this ticket before starting the next
            done += 1
        return done


started = time.perf_counter()
count = run_async(one_by_one())
one_by_one_seconds = time.perf_counter() - started
print(count, "tickets, one by one:", round(one_by_one_seconds, 1), "s")

40 tickets, one by one: 9.2 s


**Reading the number.** 40 tickets took the sum of 40 waits. For tonight's 1,000 tickets that is minutes; for 100,000 it is hours. And the computer did nothing during that time except wait for the network.

**The same job with `gather`.** The trick is to *not* wait after each ticket. First, prepare all 40 requests without sending them. Then hand the whole list to `asyncio.gather`, which sends them all and waits once.

In [3]:
async def all_at_once() -> int:
    async with AsyncOpenAI(timeout=30) as ai:
        tasks = []
        for ticket in tickets:
            tasks.append(embed_one(ai, ticket))  # prepared, not sent yet
        results = await asyncio.gather(*tasks)  # send all, wait once, results in the same order
        return len(results)


started = time.perf_counter()
count = run_async(all_at_once())
all_at_once_seconds = time.perf_counter() - started
print(count, "tickets, all at once:", round(all_at_once_seconds, 1), "s")
assert all_at_once_seconds < one_by_one_seconds

40 tickets, all at once: 1.3 s


**Reading the number.** The total dropped to about the length of one slow call: all 40 waits happened at the same time. Tonight's 1,000 tickets: well under a minute.

```
one by one    ■■ ■■ ■■ ■■ ■■ ■■ ■■ ■■ ■■ ■■ … 40 waits in a row      seconds add up
gather        ■■
              ■■   all 40 waits at the same time                       ≈ one wait
              ■■
```

**When it does not apply.** `gather` only works when the requests do not depend on each other. If ticket 2 needs the result of ticket 1, there is nothing to overlap.

| Use it when | Don't when | Instead use |
|---|---|---|
| many independent requests: embedding a batch, grading many answers, calling three providers | each step needs the previous result | `asyncio.TaskGroup` (Python 3.11+) when one failure should cancel the rest |

**Watch out**
- Sending 1,000 requests at once will hit the provider's limit and fail with "too many requests". The next item fixes that.
- If one request fails, the whole `gather` fails. Pass `return_exceptions=True` to keep the other results.
- The results come back in the same order as the list, even though they finished in a different order.